In [1]:
import pandas as pd

import random
import geopandas as gpd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics.pairwise import euclidean_distances

import sys

sys.path.insert(0, "../utils")

from modules import create_grid
from plots import *
from string_handling import *

enc = 'utf-8' 

random.seed(0)
np.random.seed(0)

pd.set_option('display.max_columns', None)

In [2]:
shapefile_path = "../../../../Shapefiles/Cameroon Shapefiles/new/"

In [3]:
cameroon_nord_lvl0 = gpd.read_file(f"{shapefile_path}Nord_region_Level0.shp",  encoding=enc)
cameroon_ext_nord_lvl0 = gpd.read_file(f"{shapefile_path}ExtremeNord_region_Level0.shp",  encoding=enc)

In [4]:
cameroon_nord_lvl3 = gpd.read_file(f"{shapefile_path}Nord_region_Level3.shp",  encoding=enc)
cameroon_ext_nord_lvl3 = gpd.read_file(f"{shapefile_path}ExtremeNord_region_Level3.shp",  encoding=enc)

In [5]:
cameroon_nord_lvl0 = cameroon_nord_lvl0.to_crs(epsg=4326)
cameroon_ext_nord_lvl0 = cameroon_ext_nord_lvl0.to_crs(epsg=4326)
cameroon_nord_lvl3 = cameroon_nord_lvl3.to_crs(epsg=4326)
cameroon_ext_nord_lvl3 = cameroon_ext_nord_lvl3.to_crs(epsg=4326)

In [6]:
cameroon_nord_lvl0.head()

,GID_1,GID_0,Pays,Nom_Region,VARNAME_1,NL_NAME_1,TYPE_1,ENGTYPE_1,CC_1,HASC_1,ISO_1,geometry
0,CMR.7_1,CMR,Cameroon,Nord,North,NA,Région,Region,NA,CM.NO,CM-NO,"POLYGON ((14.70769 7.02130, 14.70464 7.02077, ..."


In [7]:
cameroon_ext_nord_lvl0.head()

,GID_1,GID_0,Pays,Nom_Region,VARNAME_1,NL_NAME_1,TYPE_1,ENGTYPE_1,CC_1,HASC_1,ISO_1,geometry
0,CMR.4_1,CMR,Cameroon,Extrême-Nord,Extreme-North,NA,Région,Region,NA,CM.EN,NA,"POLYGON ((15.49626 10.12337, 15.49885 10.11941..."


In [8]:
cameroon_nord_lvl0.drop(columns=['GID_1','GID_0','Pays','VARNAME_1','NL_NAME_1','TYPE_1','ENGTYPE_1','CC_1','HASC_1','ISO_1'], inplace = True)
cameroon_nord_lvl0.rename(columns={"Nom_Region" : "GID0"}, inplace = True)
cameroon_nord_lvl0['GID0'] = cameroon_nord_lvl0['GID0'].str.lower()
cameroon_nord_lvl0['GID0'] = cameroon_nord_lvl0['GID0'].apply(lambda x : remove_accents(x))

cameroon_ext_nord_lvl0.drop(columns=['GID_1','GID_0','Pays','VARNAME_1','NL_NAME_1','TYPE_1','ENGTYPE_1','CC_1','HASC_1','ISO_1'], inplace = True)
cameroon_ext_nord_lvl0.rename(columns={"Nom_Region" : "GID0"}, inplace = True)
cameroon_ext_nord_lvl0['GID0'] = cameroon_ext_nord_lvl0['GID0'].str.lower()
cameroon_ext_nord_lvl0['GID0'] = cameroon_ext_nord_lvl0['GID0'].apply(lambda x : remove_accents(x))

In [9]:
cameroon_nord_lvl0.head()

,GID0,geometry
0,nord,"POLYGON ((14.70769 7.02130, 14.70464 7.02077, ..."


In [10]:
cameroon_ext_nord_lvl0.head()

,GID0,geometry
0,extreme-nord,"POLYGON ((15.49626 10.12337, 15.49885 10.11941..."


In [11]:
cameroon_nord_lvl3.head()

,OBJECTID,Nom_AS,Code_AS,District_S,Code_DS,Nom_Arrond,Code_Arron,Nom_Dept,Code_Dept,Nom_Region,Code_Reg,Id_Arrond,Pays,Code_RS,Id_DS,AVS,Autre_nom,Eff_AVS,Noms_AVS,A_statuer,No_decret,Date_creat,TITRE,Superficie,geometry
0,861,Gor,04TCH04,Tchollire,TCH,Madingring,04 04 04,Mayo-Rey,04 04,Nord,04,A185,Cameroun,NOR,182,None,None,0,None,None,None,None,GOR,6.364798e+08,"POLYGON ((14.82946 8.80523, 14.82959 8.80542, ..."
1,862,Kali,04TCH02,Tchollire,TCH,Tchollire,04 04 01,Mayo-Rey,04 04,Nord,04,A328,Cameroun,NOR,182,None,None,0,None,None,None,None,KALI,1.520400e+09,"POLYGON ((14.25497 8.47303, 14.25528 8.47297, ..."
2,863,Tchollire,04TCH01,Tchollire,TCH,Tchollire,04 04 01,Mayo-Rey,04 04,Nord,04,A328,Cameroun,NOR,182,None,None,0,None,None,None,None,TCHOLLIRE,2.045532e+09,"POLYGON ((14.10170 8.52491, 14.10191 8.52468, ..."
3,864,Gamba,04TCH09,Tchollire,TCH,Tchollire,04 04 01,Mayo-Rey,04 04,Nord,04,A328,Cameroun,NOR,182,None,None,0,None,None,None,None,GAMBA,1.525708e+09,"POLYGON ((13.73985 8.05249, 13.74000 8.05274, ..."
4,865,Sakdjé,04TCH08,Tchollire,TCH,Tchollire,04 04 01,Mayo-Rey,04 04,Nord,04,A328,Cameroun,NOR,182,None,None,0,None,None,None,None,SAKDJÉ,6.539222e+08,"POLYGON ((13.61775 8.35648, 13.61863 8.35606, ..."


In [12]:
cameroon_ext_nord_lvl3.head()

,OBJECTID,Nom_AS,Code_AS,District_S,Code_DS,Nom_Arrond,Code_Arron,Nom_Dept,Code_Dept,Nom_Region,Code_Reg,Id_Arrond,Pays,Code_RS,Id_DS,AVS,Autre_nom,Eff_AVS,Noms_AVS,A_statuer,No_decret,Date_creat,TITRE,Superficie,geometry
0,698,Damaï,09MTW01,Moutourwa,MTW,Moutourwa,09 04 05,Mayo-Kani,09 04,Extreme Nord,09,A245,Cameroun,EXT,133,None,None,0,None,None,None,None,DAMAÏ,188895834.0,"POLYGON ((14.15799 10.36594, 14.15803 10.36582..."
1,699,Moutourwa,09MTW04,Moutourwa,MTW,Moutourwa,09 04 05,Mayo-Kani,09 04,Extreme Nord,09,A245,Cameroun,EXT,133,None,None,0,None,None,None,None,MOUTOURWA,79680340.0,"POLYGON ((14.07453 10.23127, 14.07441 10.23108..."
2,701,Werfeo,09KAR13,Kar Hay,KAR,Tchatibali,09 03 10,Mayo-Danay,09 03,Extreme Nord,09,A326,Cameroun,EXT,80,None,None,0,None,None,None,None,WERFEO,145385776.0,"POLYGON ((14.83493 9.94595, 14.83488 9.94604, ..."
3,702,Guissia,09KAR08,Kar Hay,KAR,Kar Hay,09 03 02,Mayo-Danay,09 03,Extreme Nord,09,A153,Cameroun,EXT,80,None,None,0,None,None,None,None,GUISSIA,76533476.0,"POLYGON ((14.99149 10.08522, 14.99149 10.08532..."
4,703,Tchatibali,09KAR12,Kar Hay,KAR,Tchatibali,09 03 10,Mayo-Danay,09 03,Extreme Nord,09,A326,Cameroun,EXT,80,None,None,0,None,None,None,None,TCHATIBALI,59800615.0,"POLYGON ((14.94005 10.06340, 14.94025 10.06339..."


In [13]:
cameroon_nord_lvl3.drop(columns=['OBJECTID','Code_AS','Code_DS','Nom_Arrond','Code_Arron','Code_Dept','Code_Reg','Id_Arrond',
                           'Pays','Code_RS','Id_DS','AVS','Autre_nom', 'Eff_AVS', 'Noms_AVS', 'A_statuer', 'No_decret',
                            'Date_creat', 'TITRE', 'Superficie'], inplace = True)
cameroon_nord_lvl3.rename(columns={"Nom_AS": "GID3", "District_S": "GID2", "Nom_Dept": "GID1", "Nom_Region" : "GID0"}, inplace = True)
cameroon_nord_lvl3['GID0'] = cameroon_nord_lvl3['GID0'].str.lower()
cameroon_nord_lvl3['GID0'] = cameroon_nord_lvl3['GID0'].apply(lambda x : remove_accents(x))
cameroon_nord_lvl3['GID1'] = cameroon_nord_lvl3['GID1'].str.lower()
cameroon_nord_lvl3['GID1'] = cameroon_nord_lvl3['GID1'].apply(lambda x : remove_accents(x))
cameroon_nord_lvl3['GID2'] = cameroon_nord_lvl3['GID2'].str.lower()
cameroon_nord_lvl3['GID2'] = cameroon_nord_lvl3['GID2'].apply(lambda x : remove_accents(x))
cameroon_nord_lvl3['GID3'] = cameroon_nord_lvl3['GID3'].str.lower()
cameroon_nord_lvl3['GID3'] = cameroon_nord_lvl3['GID3'].apply(lambda x : remove_accents(x))

cameroon_ext_nord_lvl3.drop(columns=['OBJECTID','Code_AS','Code_DS','Nom_Arrond','Code_Arron','Code_Dept','Code_Reg','Id_Arrond',
                           'Pays','Code_RS','Id_DS','AVS','Autre_nom', 'Eff_AVS', 'Noms_AVS', 'A_statuer', 'No_decret',
                            'Date_creat', 'TITRE', 'Superficie'], inplace = True)
cameroon_ext_nord_lvl3.rename(columns={"Nom_AS": "GID3", "District_S": "GID2", "Nom_Dept": "GID1", "Nom_Region" : "GID0"}, inplace = True)
cameroon_ext_nord_lvl3['GID0'] = cameroon_ext_nord_lvl3['GID0'].str.lower()
cameroon_ext_nord_lvl3['GID0'] = cameroon_ext_nord_lvl3['GID0'].apply(lambda x : remove_accents(x))
cameroon_ext_nord_lvl3['GID1'] = cameroon_ext_nord_lvl3['GID1'].str.lower()
cameroon_ext_nord_lvl3['GID1'] = cameroon_ext_nord_lvl3['GID1'].apply(lambda x : remove_accents(x))
cameroon_ext_nord_lvl3['GID2'] = cameroon_ext_nord_lvl3['GID2'].str.lower()
cameroon_ext_nord_lvl3['GID2'] = cameroon_ext_nord_lvl3['GID2'].apply(lambda x : remove_accents(x))
cameroon_ext_nord_lvl3['GID3'] = cameroon_ext_nord_lvl3['GID3'].str.lower()
cameroon_ext_nord_lvl3['GID3'] = cameroon_ext_nord_lvl3['GID3'].apply(lambda x : remove_accents(x))

In [14]:
cameroon_nord_lvl3.head()

,GID3,GID2,GID1,GID0,geometry
0,gor,tchollire,mayo-rey,nord,"POLYGON ((14.82946 8.80523, 14.82959 8.80542, ..."
1,kali,tchollire,mayo-rey,nord,"POLYGON ((14.25497 8.47303, 14.25528 8.47297, ..."
2,tchollire,tchollire,mayo-rey,nord,"POLYGON ((14.10170 8.52491, 14.10191 8.52468, ..."
3,gamba,tchollire,mayo-rey,nord,"POLYGON ((13.73985 8.05249, 13.74000 8.05274, ..."
4,sakdje,tchollire,mayo-rey,nord,"POLYGON ((13.61775 8.35648, 13.61863 8.35606, ..."


In [15]:
cameroon_ext_nord_lvl3.head()

,GID3,GID2,GID1,GID0,geometry
0,damai,moutourwa,mayo-kani,extreme nord,"POLYGON ((14.15799 10.36594, 14.15803 10.36582..."
1,moutourwa,moutourwa,mayo-kani,extreme nord,"POLYGON ((14.07453 10.23127, 14.07441 10.23108..."
2,werfeo,kar hay,mayo-danay,extreme nord,"POLYGON ((14.83493 9.94595, 14.83488 9.94604, ..."
3,guissia,kar hay,mayo-danay,extreme nord,"POLYGON ((14.99149 10.08522, 14.99149 10.08532..."
4,tchatibali,kar hay,mayo-danay,extreme nord,"POLYGON ((14.94005 10.06340, 14.94025 10.06339..."


In [16]:
cameroon_nord_lvl0.to_file(f'{shapefile_path}web_platform/CM_Nord_LV0.shp', encoding = enc)
cameroon_ext_nord_lvl0.to_file(f'{shapefile_path}web_platform/CM_ExtremeNord_LV0.shp', encoding = enc)
cameroon_nord_lvl3.to_file(f'{shapefile_path}web_platform/CM_Nord_LV3.shp', encoding = enc)
cameroon_ext_nord_lvl3.to_file(f'{shapefile_path}web_platform/CM_ExtremeNord_LV3.shp', encoding = enc)